# Ray Concepts Demo

Step-by-step walkthrough of all Ray modules in this project:

1. **Core Primitives** – remote tasks, actors, object store
2. **Ray Data** – preprocessing pipeline
3. **Ray Train** – distributed DDP training
4. **Ray Tune** – hyperparameter optimisation
5. **Ray Serve** – multi-model inference pipeline

> **Prerequisite**: `pip install ray[all] torch pyyaml numpy pandas`

In [ ]:
import sys
import os
from pathlib import Path

# Add the project src to path
PROJECT_ROOT = Path("../..")
SRC_PATH = str(PROJECT_ROOT / "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

CONFIG_PATH = str(PROJECT_ROOT / "config.yaml")
print(f"Project root: {PROJECT_ROOT.resolve()}")
print(f"Config: {CONFIG_PATH}")

In [ ]:
from utils.config_loader import load_config
from utils.logging_setup import get_logger

config = load_config(CONFIG_PATH)
logger = get_logger("notebook", config)
logger.info("Notebook started")

import pprint
pprint.pprint(config)

---
## 1. Core Primitives
### 1a. Remote Tasks

In [ ]:
import ray
import time
import numpy as np

ray_cfg = config["ray"]["init"]
ray.init(
    num_cpus=ray_cfg["num_cpus"],
    num_gpus=ray_cfg["num_gpus"],
    object_store_memory=ray_cfg["object_store_memory"],
    ignore_reinit_error=True,
)
print("Ray resources:", ray.cluster_resources())

In [ ]:
@ray.remote
def square(x):
    return x * x

# Correct pattern: submit all, retrieve once
batch_size = config["core_primitives"]["batch_size"]
futures = [square.remote(i) for i in range(batch_size)]
results = ray.get(futures)

print(f"Submitted {batch_size} tasks")
print(f"First 5 results: {results[:5]}")
print(f"Last 5 results:  {results[-5:]}")

In [ ]:
# ray.wait() streaming demo
import random

@ray.remote
def slow_task(task_id, secs):
    time.sleep(secs)
    return {"task_id": task_id, "sleep": secs}

sleep_times = [round(random.uniform(0.05, 0.3), 2) for _ in range(6)]
futures = [slow_task.remote(i, t) for i, t in enumerate(sleep_times)]

remaining = futures.copy()
order_completed = []
while remaining:
    ready, remaining = ray.wait(remaining, num_returns=1, timeout=1.0)
    if ready:
        result = ray.get(ready[0])
        order_completed.append(result)
        print(f"Completed: {result}  ({len(remaining)} remaining)")

### 1b. Actors

In [ ]:
@ray.remote
class Counter:
    def __init__(self):
        self._n = 0
    def increment(self):
        self._n += 1
        return self._n
    def value(self):
        return self._n

counter = Counter.remote()

# Batch futures (correct pattern)
futures = [counter.increment.remote() for _ in range(10)]
values = ray.get(futures)
print("Values after 10 increments:", values)
print("Final state:", ray.get(counter.value.remote()))

In [ ]:
# ParameterServer pattern
@ray.remote
class ParameterServer:
    def __init__(self, dim, lr):
        self.params = np.zeros(dim, dtype=np.float32)
        self.lr = lr
        self.updates = 0

    def apply_gradients(self, *grads):
        avg = np.mean(np.stack(grads), axis=0)
        self.params -= self.lr * avg
        self.updates += 1
        return self.params.copy()

    def get_params(self):
        return self.params.copy()

@ray.remote
class Worker:
    def __init__(self, wid, dim):
        self.wid = wid
        self.rng = np.random.default_rng(wid)

    def compute_gradient(self, params, step):
        scale = 1.0 / (1 + step * 0.1)
        return self.rng.normal(0, scale, params.shape).astype(np.float32)

lr = config["distributed_training"]["learning_rate"]
num_workers = config["core_primitives"]["num_workers"]
dim = 32

ps = ParameterServer.remote(dim, lr)
workers = [Worker.remote(i, dim) for i in range(num_workers)]

param_norms = []
for step in range(5):
    params = ray.get(ps.get_params.remote())
    grads = ray.get([w.compute_gradient.remote(params, step) for w in workers])
    new_params = ray.get(ps.apply_gradients.remote(*grads))
    norm = float(np.linalg.norm(new_params))
    param_norms.append(norm)
    print(f"Step {step+1}: param_norm={norm:.4f}")

print("\nParam norm over steps:", [round(n, 4) for n in param_norms])

### 1c. Object Store

In [ ]:
# ray.put() – store once, read many times
large_data = np.random.standard_normal(10_000).astype(np.float32)

@ray.remote
def process_slice(data, start, end):
    return float(data[start:end].sum())

# Put once → all 4 tasks share the same plasma buffer
data_ref = ray.put(large_data)
print(f"Object ref type: {type(data_ref)}")
print(f"Object ref: {data_ref}")

chunk = len(large_data) // 4
futures = [
    process_slice.remote(data_ref, i*chunk, (i+1)*chunk)
    for i in range(4)
]
partial_sums = ray.get(futures)
total = sum(partial_sums)

print(f"Partial sums: {[round(s,4) for s in partial_sums]}")
print(f"Total (distributed): {total:.4f}")
print(f"Total (numpy):       {large_data.sum():.4f}")

---
## 2. Ray Data Pipeline

In [ ]:
import ray.data
import pandas as pd

from distributed_training.data_pipeline import DataPipelineBuilder

builder = DataPipelineBuilder(num_samples=800, num_features=10)
raw_ds = builder.build_dataset()
print("Raw dataset schema:", raw_ds.schema())
print("Row count:", raw_ds.count())

In [ ]:
processed_ds = builder.apply_preprocessing(raw_ds)
train_ds, val_ds = builder.split_train_val(processed_ds, val_fraction=0.2)

print(f"Train rows: {train_ds.count()}")
print(f"Val rows:   {val_ds.count()}")

sample = train_ds.take_batch(batch_size=5, batch_format="pandas")
print("\nSample batch (first 5 rows):")
sample

In [ ]:
# Visualise label distribution
import matplotlib.pyplot as plt

full_sample = train_ds.take_batch(batch_size=640, batch_format="pandas")
label_counts = full_sample["label"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Class 0", "Class 1"], label_counts.values, color=["steelblue", "tomato"])
ax.set_title("Training Set Label Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig("label_distribution.png", dpi=120)
plt.show()
print("Saved label_distribution.png")

---
## 3. Ray Train – Distributed PyTorch

In [ ]:
from distributed_training.torch_trainer import DistributedTrainerFactory

train_loop_config = {
    "epochs": config["distributed_training"]["epochs"],
    "learning_rate": config["distributed_training"]["learning_rate"],
    "batch_size": config["distributed_training"]["batch_size"],
    "hidden_dim": 64,
    "dropout_rate": 0.2,
    "num_samples": 1500,
}

trainer = DistributedTrainerFactory.build(
    num_workers=config["distributed_training"]["num_workers"],
    use_gpu=config["distributed_training"]["use_gpu"],
    train_loop_config=train_loop_config,
    num_to_keep=config["distributed_training"]["num_to_keep_checkpoints"],
)
print("TorchTrainer built:", trainer)

In [ ]:
result = trainer.fit()
print("\nTraining metrics:", result.metrics)

# Plot loss curve
metrics_df = result.metrics_dataframe
if metrics_df is not None and "loss" in metrics_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(metrics_df["epoch"], metrics_df["loss"], marker="o", color="tomato")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE Loss")

    axes[1].plot(metrics_df["epoch"], metrics_df["accuracy"], marker="s", color="steelblue")
    axes[1].set_title("Training Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")

    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=120)
    plt.show()
    print("Saved training_curves.png")

---
## 4. Ray Tune – Hyperparameter Search

In [ ]:
from hyperparameter_tuning.tune_experiment import TuneExperimentRunner, build_param_space

tune_cfg = config["hyperparameter_tuning"]
print("Param space:")
import pprint
pprint.pprint(build_param_space())

In [ ]:
# Run a small experiment (reduce num_samples for notebook speed)
runner = TuneExperimentRunner(
    num_samples=4,  # small for demo; config has 20
    metric=tune_cfg["metric"],
    mode=tune_cfg["mode"],
    grace_period=tune_cfg["grace_period"],
    reduction_factor=tune_cfg["reduction_factor"],
)
results = runner.run()

best = results.get_best_result(metric="accuracy", mode="max")
print("\nBest trial config:")
pprint.pprint(best.config)
print("Best metrics:", best.metrics)

In [ ]:
# Visualise search results
df = results.get_dataframe()
print(df.columns.tolist())

if "accuracy" in df.columns and "config/train_loop_config/lr" in df.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    sc = ax.scatter(
        df["config/train_loop_config/lr"],
        df["accuracy"],
        c=df["config/train_loop_config/hidden_dim"],
        cmap="viridis",
        s=100,
        edgecolors="k",
    )
    plt.colorbar(sc, label="Hidden Dim")
    ax.set_xscale("log")
    ax.set_xlabel("Learning Rate (log scale)")
    ax.set_ylabel("Accuracy")
    ax.set_title("Hyperparameter Search Results")
    plt.tight_layout()
    plt.savefig("hpo_results.png", dpi=120)
    plt.show()
    print("Saved hpo_results.png")

---
## 5. Ray Serve – Multi-Model Pipeline

In [ ]:
import asyncio
from model_serving.serve_deployment import ServeDeploymentManager, run_test_requests

manager = ServeDeploymentManager()
handle = manager.start()
print("Deployments running")

In [ ]:
# Send requests via the Python handle
serve_cfg = config["model_serving"]
await run_test_requests(handle, num_requests=10)

In [ ]:
# Visualise latency distribution
import time

rng = np.random.default_rng(seed=55)
latencies = []

async def timed_request():
    features = rng.standard_normal(10).tolist()
    t0 = time.perf_counter()
    result = await handle.remote({"features": features})
    return (time.perf_counter() - t0) * 1000

latencies = await asyncio.gather(*[timed_request() for _ in range(30)])
latencies = list(latencies)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(latencies, bins=15, color="mediumseagreen", edgecolor="k")
ax.axvline(np.mean(latencies), color="red", linestyle="--", label=f"Mean: {np.mean(latencies):.1f}ms")
ax.set_xlabel("Latency (ms)")
ax.set_ylabel("Count")
ax.set_title("Request Latency Distribution (Ray Serve Pipeline)")
ax.legend()
plt.tight_layout()
plt.savefig("serving_latency.png", dpi=120)
plt.show()
print(f"Saved serving_latency.png | p50={np.percentile(latencies,50):.1f}ms p99={np.percentile(latencies,99):.1f}ms")

In [ ]:
# Cleanup
manager.shutdown()
ray.shutdown()
print("All resources released.")

---
## Summary

| Module | Key API | Pattern |
|--------|---------|--------|
| Core – Tasks | `@ray.remote`, `ray.get`, `ray.wait` | Submit-all-then-get |
| Core – Actors | `@ray.remote class`, `ActorPool` | ParameterServer, StatefulWorker |
| Core – Object Store | `ray.put`, `ObjectRef` | Fan-out shared data |
| Ray Data | `map_batches`, `split_at_indices` | ETL preprocessing chain |
| Ray Train | `TorchTrainer`, `ScalingConfig`, `Checkpoint` | DDP with checkpointing |
| Ray Tune | `Tuner`, `ASHAScheduler`, `OptunaSearch` | Bayesian HPO + early stopping |
| Ray Serve | `@serve.deployment`, `@serve.batch` | Multi-model batched pipeline |